In [1]:
from chi import server, context, lease, network
import chi, os, time, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

In [2]:
LEASE_NAME = "proj27-mlops-devops"

l = lease.Lease(
    LEASE_NAME,
    duration=datetime.timedelta(hours=8),
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.large"),
    amount=1,
)
l.submit(idempotent=True)
l.show()

Waiting for lease to start...


Lease proj27-mlops-devops has reached status active


HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>proj27-mlops-d…

Lease Details:
Name: proj27-mlops-devops
ID: 22cdea8d-5dde-46f1-9d3d-184879ef43b7
Status: ACTIVE
Start Date: 2026-04-03 13:17:00
End Date: 2026-04-03 21:17:00
User ID: 4eade18a2ae0994011ddd200e76ed5f53af817042cb649ace44d56f6194a51eb
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: 6418a781-3eaa-4019-a4ae-8e37e0581aa5, Status: active, Flavor: 6418a781-3eaa-4019-a4ae-8e37e0581aa5, Amount: 1

Events:


In [3]:
SERVER_NAME = "proj27-k8s-node"

s = server.Server(
    SERVER_NAME,
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server proj27-k8s-node's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,proj27-k8s-node
Id,a5c85e86-a47b-4ee1-ab33-5431081ab09a
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:6418a781-3eaa-4019-a4ae-8e37e0581aa5
Addresses,sharednet1: IP: 10.56.2.170 (v4) Type: fixed MAC: fa:16:3e:e3:8d:a8
Network Name,sharednet1
Created At,2026-04-03T13:17:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,4f8e0835c495f995792aecd839e934f9dd1201938fdb2269b5562782


In [4]:
s.refresh()
s.show(type="widget")

Attribute,proj27-k8s-node
Id,a5c85e86-a47b-4ee1-ab33-5431081ab09a
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:6418a781-3eaa-4019-a4ae-8e37e0581aa5
Addresses,sharednet1: IP: 10.56.2.170 (v4) Type: fixed MAC: fa:16:3e:e3:8d:a8
Network Name,sharednet1
Created At,2026-04-03T13:17:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,4f8e0835c495f995792aecd839e934f9dd1201938fdb2269b5562782


In [5]:
s.associate_floating_ip()

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


'129.114.27.244'

In [6]:
s.refresh()
s.check_connectivity()
s.show(type="widget")

Checking connectivity to 129.114.27.244 port 22.


Connection successful


Attribute,proj27-k8s-node
Id,a5c85e86-a47b-4ee1-ab33-5431081ab09a
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:6418a781-3eaa-4019-a4ae-8e37e0581aa5
Addresses,sharednet1: IP: 10.56.2.170 (v4) Type: fixed MAC: fa:16:3e:e3:8d:a8 IP: 129.114.27.244 (v4) Type: floating MAC: fa:16:3e:e3:8d:a8
Network Name,sharednet1
Created At,2026-04-03T13:17:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,4f8e0835c495f995792aecd839e934f9dd1201938fdb2269b5562782


In [7]:
security_groups = [
  {'name': "allow-ssh", "protocol": "tcp", 'port': 22, 'description': "Enable SSH traffic on TCP port 22"},
  {'name': "allow-8888", "protocol": "tcp", 'port': 8888, 'description': "Enable TCP port 8888 (used by Jupyter)"},
  {"name": "allow-30080", "protocol": "tcp", "port": 30080, "description": "Enable Jitsi Web on TCP port 30080"},
  {"name": "allow-jvb-10000-udp", "port": 10000, "protocol": "udp", "description": "Enable Jitsi JVB media on UDP port 10000"},
  {"name": "allow-jvb-30000-udp", "port": 30000, "protocol": "udp", "description": "Enable Jitsi JVB media on UDP port 30000"},
  {"name": "allow-30500", "port": 30500, "protocol": "tcp", "description": "Enable MLflow UI on TCP port 30500"},
  {"name": "allow-http-80", "port": 80, "protocol": "tcp", "description": "Enable HTTP on TCP 80"},
  {"name": "allow-https-443", "port": 443, "protocol": "tcp", "description": "Enable HTTPS on TCP 443"},
]

In [8]:
for sg in security_groups:
  secgroup = network.SecurityGroup({
      'name': sg['name'],
      'description': sg['description'],
  })
  secgroup.add_rule(direction='ingress', protocol=sg["protocol"], port=sg['port'])
  secgroup.submit(idempotent=True)
  s.add_security_group(sg['name'])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The py

updated security groups: ['allow-ssh', 'allow-8888', 'allow-30080', 'allow-jvb-10000-udp', 'allow-jvb-30000-udp', 'allow-30500', 'allow-http-80', 'allow-https-443']


In [9]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.27.244 port 22.


Connection successful


Use local terminal to ssh (substitute `A.B.C.D` with your `floating ip`:
```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```